# 05 — Train an auxiliary EGFR mutation-effect model

This notebook uses the large EGFR kinase-domain mutation dataset to learn a **general mutation-effect score**.

It does **not** treat this score as drug resistance. Instead, the trained model will later provide an additional feature to the smaller EGFR TKI-resistance model.

## Inputs

Upload:

- `train.csv`
- `test.csv`
- `egfr-dfianalysisMaxAll_kinase.csv`

## Outputs

- held-out test metrics,
- predicted mutation-effect scores,
- feature importance,
- a fitted reusable model,
- `egfr_auxiliary_model_bundle.zip`.

## Scientific role

The large dataset teaches the model which substitutions tend to strongly perturb the EGFR kinase domain. The smaller Hayes dataset will then learn whether that general perturbation contributes to drug-specific resistance.

## Step 1 — Upload the three input files

In [ ]:
from google.colab import files

uploaded = files.upload()
print("Uploaded:")
for name in uploaded:
    print(" -", name)

## Step 2 — Install required packages

In [ ]:
!pip -q install pandas numpy scikit-learn scipy matplotlib joblib

## Step 3 — Load and inspect the mutation datasets

The provided train/test split is preserved. The `score` column is treated as an auxiliary mutation-effect outcome, not as a resistance label.

In [ ]:
import pandas as pd
import numpy as np

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

dfi_candidates = [
    "egfr-dfianalysisMaxAll_kinase.csv",
    "egfr-dfianalysisMaxAll_kinase(1).csv",
]
dfi_file = next((name for name in dfi_candidates if name in uploaded), None)
if dfi_file is None:
    raise FileNotFoundError(
        "Upload egfr-dfianalysisMaxAll_kinase.csv"
    )

dfi = pd.read_csv(dfi_file)

required_mutation_columns = {"variant", "score"}
for label, frame in [("train", train), ("test", test)]:
    missing = required_mutation_columns.difference(frame.columns)
    if missing:
        raise ValueError(f"{label}.csv missing columns: {sorted(missing)}")

required_dfi_columns = {"ResI", "DFI", "DFI%", "minus_ln_pctdfi"}
missing_dfi = required_dfi_columns.difference(dfi.columns)
if missing_dfi:
    raise ValueError(f"DFI file missing columns: {sorted(missing_dfi)}")

print("Training variants:", len(train))
print("Held-out test variants:", len(test))
print("DFI residues:", len(dfi))
print("Training score range:", train["score"].min(), "to", train["score"].max())
print("Test score range:", test["score"].min(), "to", test["score"].max())

display(train[["variant", "score"]].head())
display(dfi.head())

## Step 4 — Convert mutation notation into biologically meaningful features

For a variant such as `T790M`, the model receives:

- original amino acid,
- residue position,
- mutant amino acid,
- side-chain volume change,
- hydropathy change,
- charge change,
- aromaticity and polarity changes,
- DFI at the mutation site,
- DFI percentile and rigidity score.

This is more compact and interpretable than using 267 separate sequence columns.

In [ ]:
import re

AA_VOLUME = {
    "A": 88.6, "R": 173.4, "N": 114.1, "D": 111.1, "C": 108.5,
    "Q": 143.8, "E": 138.4, "G": 60.1, "H": 153.2, "I": 166.7,
    "L": 166.7, "K": 168.6, "M": 162.9, "F": 189.9, "P": 112.7,
    "S": 89.0, "T": 116.1, "W": 227.8, "Y": 193.6, "V": 140.0,
}

AA_HYDROPATHY = {
    "A": 1.8, "R": -4.5, "N": -3.5, "D": -3.5, "C": 2.5,
    "Q": -3.5, "E": -3.5, "G": -0.4, "H": -3.2, "I": 4.5,
    "L": 3.8, "K": -3.9, "M": 1.9, "F": 2.8, "P": -1.6,
    "S": -0.8, "T": -0.7, "W": -0.9, "Y": -1.3, "V": 4.2,
}

AA_CHARGE = {
    "A": 0.0, "R": 1.0, "N": 0.0, "D": -1.0, "C": 0.0,
    "Q": 0.0, "E": -1.0, "G": 0.0, "H": 0.1, "I": 0.0,
    "L": 0.0, "K": 1.0, "M": 0.0, "F": 0.0, "P": 0.0,
    "S": 0.0, "T": 0.0, "W": 0.0, "Y": 0.0, "V": 0.0,
}

AA_AROMATIC = {aa: int(aa in {"F", "W", "Y", "H"}) for aa in AA_VOLUME}
AA_POLAR = {
    aa: int(aa in {"R", "N", "D", "Q", "E", "H", "K", "S", "T", "Y", "C"})
    for aa in AA_VOLUME
}

def parse_variant(variant):
    match = re.fullmatch(r"([A-Z])(\d+)([A-Z])", str(variant).strip().upper())
    if not match:
        raise ValueError(f"Unsupported variant notation: {variant}")
    wt, position, mutant = match.groups()
    return wt, int(position), mutant

dfi_lookup = dfi.set_index("ResI").to_dict("index")

def make_variant_features(frame):
    rows = []

    for row in frame.itertuples(index=False):
        wt, position, mutant = parse_variant(row.variant)
        dfi_row = dfi_lookup.get(position, {})

        rows.append({
            "variant": row.variant,
            "score": float(row.score),
            "wt_aa": wt,
            "mut_aa": mutant,
            "position": position,
            "position_scaled": (position - 712) / (978 - 712),

            "delta_sidechain_volume_A3":
                AA_VOLUME[mutant] - AA_VOLUME[wt],
            "absolute_volume_change_A3":
                abs(AA_VOLUME[mutant] - AA_VOLUME[wt]),
            "delta_hydropathy":
                AA_HYDROPATHY[mutant] - AA_HYDROPATHY[wt],
            "absolute_hydropathy_change":
                abs(AA_HYDROPATHY[mutant] - AA_HYDROPATHY[wt]),
            "delta_charge":
                AA_CHARGE[mutant] - AA_CHARGE[wt],
            "charge_changed":
                int(AA_CHARGE[mutant] != AA_CHARGE[wt]),
            "delta_aromatic":
                AA_AROMATIC[mutant] - AA_AROMATIC[wt],
            "delta_polar":
                AA_POLAR[mutant] - AA_POLAR[wt],

            "site_DFI": dfi_row.get("DFI", np.nan),
            "site_DFI_percentile": dfi_row.get("DFI%", np.nan),
            "site_rigidity_score": dfi_row.get("minus_ln_pctdfi", np.nan),
        })

    return pd.DataFrame(rows)

train_features = make_variant_features(train)
test_features = make_variant_features(test)

print("Training feature table:", train_features.shape)
print("Test feature table:", test_features.shape)
display(train_features.head())

## Step 5 — Build the auxiliary regression model

The model combines:

- one-hot encoded original and mutant amino acids,
- numeric physicochemical changes,
- residue position,
- DFI-derived site properties.

A histogram gradient-boosting regressor is used because it handles nonlinear effects efficiently on a few thousand examples.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingRegressor

categorical_features = ["wt_aa", "mut_aa"]
numeric_features = [
    "position",
    "position_scaled",
    "delta_sidechain_volume_A3",
    "absolute_volume_change_A3",
    "delta_hydropathy",
    "absolute_hydropathy_change",
    "delta_charge",
    "charge_changed",
    "delta_aromatic",
    "delta_polar",
    "site_DFI",
    "site_DFI_percentile",
    "site_rigidity_score",
]

preprocessor = ColumnTransformer([
    (
        "numeric",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]),
        numeric_features,
    ),
    (
        "amino_acids",
        OneHotEncoder(handle_unknown="ignore", sparse_output=False),
        categorical_features,
    ),
])

aux_model = Pipeline([
    ("preprocess", preprocessor),
    ("model", HistGradientBoostingRegressor(
        learning_rate=0.05,
        max_iter=400,
        max_leaf_nodes=20,
        min_samples_leaf=15,
        l2_regularization=1.0,
        random_state=42,
    )),
])

X_train = train_features[numeric_features + categorical_features]
y_train = train_features["score"]

X_test = test_features[numeric_features + categorical_features]
y_test = test_features["score"]

aux_model.fit(X_train, y_train)
print("Auxiliary mutation-effect model fitted.")

## Step 6 — Evaluate on the provided held-out test set

The test set was not used during fitting.

Metrics:

- MAE: average absolute error,
- RMSE: emphasizes large errors,
- R²: improvement over always predicting the training-set mean,
- Spearman correlation: whether the model correctly ranks stronger versus weaker mutation effects.


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import spearmanr

test_pred = aux_model.predict(X_test)

metrics = {
    "train_rows": int(len(train_features)),
    "test_rows": int(len(test_features)),
    "MAE": float(mean_absolute_error(y_test, test_pred)),
    "RMSE": float(mean_squared_error(y_test, test_pred) ** 0.5),
    "R2": float(r2_score(y_test, test_pred)),
    "Spearman_r": float(spearmanr(y_test, test_pred).statistic),
    "training_score_mean_baseline": float(y_train.mean()),
}

print(metrics)

test_predictions = test_features[["variant", "score"]].copy()
test_predictions["predicted_score"] = test_pred
test_predictions["absolute_error"] = abs(
    test_predictions["score"] - test_predictions["predicted_score"]
)

display(
    test_predictions.sort_values("absolute_error", ascending=False).head(10)
)

## Step 7 — Plot observed versus predicted mutation-effect score

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 6))
plt.scatter(y_test, test_pred, alpha=0.6)

low = min(y_test.min(), test_pred.min())
high = max(y_test.max(), test_pred.max())
plt.plot([low, high], [low, high], linestyle="--")

plt.xlabel("Observed mutation-effect score")
plt.ylabel("Predicted mutation-effect score")
plt.title("Auxiliary EGFR model: held-out test performance")
plt.tight_layout()
plt.show()

## Step 8 — Calculate permutation importance

This identifies which compact mutation and DFI descriptors are most useful on the held-out test set.

In [ ]:
from sklearn.inspection import permutation_importance

permutation = permutation_importance(
    aux_model,
    X_test,
    y_test,
    scoring="neg_mean_absolute_error",
    n_repeats=20,
    random_state=42,
    n_jobs=-1,
)

importance = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": permutation.importances_mean,
    "importance_std": permutation.importances_std,
}).sort_values("importance_mean", ascending=False)

display(importance)

## Step 9 — Fit a final model using all 4,802 scored variants

After unbiased evaluation on the original held-out test set, the final deployable auxiliary model is refitted on the combined train and test datasets.

In [ ]:
all_features = pd.concat(
    [train_features, test_features],
    ignore_index=True,
)

X_all = all_features[numeric_features + categorical_features]
y_all = all_features["score"]

final_aux_model = Pipeline([
    ("preprocess", preprocessor),
    ("model", HistGradientBoostingRegressor(
        learning_rate=0.05,
        max_iter=400,
        max_leaf_nodes=20,
        min_samples_leaf=15,
        l2_regularization=1.0,
        random_state=42,
    )),
])

final_aux_model.fit(X_all, y_all)

all_features["auxiliary_predicted_score"] = final_aux_model.predict(X_all)

print("Final model fitted on:", len(all_features), "variants")
display(all_features[["variant", "score", "auxiliary_predicted_score"]].head())

## Step 10 — Save the model bundle

The bundle contains:

- fitted auxiliary model,
- DFI table,
- feature definitions,
- test metrics,
- held-out predictions,
- feature importance,
- predictions for all 4,802 variants.

We will use this bundle in the final rich-feature integration notebook.

In [ ]:
import json
import joblib
import shutil
from pathlib import Path

output_dir = Path("egfr_auxiliary_model")
if output_dir.exists():
    shutil.rmtree(output_dir)
output_dir.mkdir()

joblib.dump(final_aux_model, output_dir / "auxiliary_mutation_effect_model.joblib")
dfi.to_csv(output_dir / "egfr_dfi_profile.csv", index=False)
test_predictions.to_csv(output_dir / "heldout_test_predictions.csv", index=False)
importance.to_csv(output_dir / "auxiliary_feature_importance.csv", index=False)
all_features[
    ["variant", "score", "auxiliary_predicted_score"]
].to_csv(output_dir / "all_variant_predictions.csv", index=False)

with open(output_dir / "auxiliary_metrics.json", "w") as handle:
    json.dump(metrics, handle, indent=2)

with open(output_dir / "feature_schema.json", "w") as handle:
    json.dump({
        "numeric_features": numeric_features,
        "categorical_features": categorical_features,
        "note": (
            "Auxiliary score is a general EGFR mutation-effect prediction, "
            "not a drug-resistance label."
        ),
    }, handle, indent=2)

archive = shutil.make_archive(
    "egfr_auxiliary_model_bundle",
    "zip",
    root_dir=output_dir,
)

print("Created:", archive)
files.download(archive)

## Next step

The next notebook will combine:

- enhanced structural features,
- drug descriptors,
- interaction-change features,
- DFI,
- the auxiliary predicted mutation-effect score,

and then retrain the drug-specific Hayes resistance model.